<a href="https://colab.research.google.com/github/Innej3/PatientZero/blob/main/PatientZero-SEIR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
'''
Changes from SIR model

S->E->I->R. So infeccted neigbors now become E
E->I with prob sigma per step
E is hidden. We only observe I and R (same as in real world)

'''

In [ ]:
S, E_, I_, R_ = 0, 1, 2, 3

In [ ]:
def generate_topology(case='BA', n=90, seed=42):
  rng = np.random.default_rng(seed)
  if case == 'ER':
    G = nx.erdos_renyi_graph(n, p=0.065, seed=seed)
  elif case == 'BA':
    G = nx.barabasi_albert_graph(n, 2, seed=seed)
  elif case == 'WS':
    G = nx.watts_strogatz_graph(n, 4, 0.15, seed=seed)
  else:
    raise ValueError('Invalid TOPO')

  if not nx.is_connected(G):
    largest_cc = max(nx.connected_components(G), key=len)
    G = G.subgraph(largest_cc).copy()
    G = nx.convert_node_labels_to_integers(G)

  degrees = dict(G.degree())
  max_hub = max(degrees, key=degrees.get)


  if case == 'BA':
    lengths = nx.single_source_shortest_path_length(G, max_hub)
    candidates = [u for u,d in lengths.items() if d==2 and degrees[u] <= 2]
    patient_zero = candidates[0] if candidates else rng.choice(list(G.nodes()))
  else:
    patient_zero = rng.choice(list(G.nodes()))

  pos = nx.spring_layout(G, k=0.38, iterations=80, seed=seed)
  return G, pos, patient_zero, max_hub

In [ ]:
def simulate_seir(adj, n, source, beta, sigma, gamma, target_frac, rng, max_steps=400):
  """
  Synchronous SEIR. Stops at the first step where visible cases (I + R) >= target.
  Returns the state array, or None if the outbreak dies out first.
  """
  state = np.zeros(n, dtype=int)
  state[source] = I_
  target = int(target_frac * n)
  for _ in range(max_steps):
    if ((state == I_) | (state == R_)).sum() >= target:
      return state
    inf = np.flatnonzero(state == I_)
    exp = np.flatnonzero(state == E_)
    if len(inf) == 0 and len(exp) == 0:
      return None
    new_exp = set()
    for u in inf:
      for v in adj[u]:
        if state[v] == S and rng.random() < beta:
          new_exp.add(v)
    become_inf = exp[rng.random(len(exp)) < sigma]    # only nodes exposed before this step
    recover = inf[rng.random(len(inf)) < gamma]        # only nodes infectious before this step
    if new_exp:
      state[list(new_exp)] = E_
    state[become_inf] = I_
    state[recover] = R_
  return None


In [ ]:
def jaccard(a, b):
  if not a and not b:
    return 1.0
  return len(a & b) / len(a | b)

In [ ]:
def abc_seir(adj, n, obs_I, obs_R, mode, candidates, beta, sigma, gamma, target_frac,
             n_trials, rng, eps=0.12):
  scores = np.zeros(n)
  for c in candidates:
    acc = 0.0
    for _ in range(n_trials):
      st = simulate_seir(adj, n, c, beta, sigma, gamma, target_frac, rng)
      if st is None:
        continue
      sI = set(np.flatnonzero(st == I_).tolist())      # E is hidden in simulations too
      sR = set(np.flatnonzero(st == R_).tolist())
      if mode == 'I':
        J = jaccard(sI, obs_I)
      elif mode == 'IR':
        J = jaccard(sI | sR, obs_I | obs_R)
      else:
        parts = [jaccard(sI, obs_I)]
        if obs_R or sR:
          parts.append(jaccard(sR, obs_R))
        J = np.mean(parts)
      acc += np.exp((J - 1.0) / eps)
    scores[c] = acc / n_trials
  post = np.zeros(n)
  if scores.sum() > 0:
    post = scores / scores.sum()
  else:
    post[list(candidates)] = 1.0 / len(candidates)
  return post


In [ ]:
def tie_rank(post, src):
  p = post[src]
  higher = int((post > p + 1e-12).sum())
  ties = int((np.abs(post - p) <= 1e-12).sum())
  return higher + (ties + 1) / 2

In [ ]:
def jordan_posterior(G, infected_set, tau=0.6):

  G_I = G.subgraph(infected_set)
  post = np.zeros(len(G))
  if len(infected_set) == 1:
    only_node = next(iter(infected_set))
    post[only_node] = 1.0
    return post

  ecc = {}
  for v in infected_set:
    lengths = nx.single_source_shortest_path_length(G_I, v)
    ecc[v] = max(lengths.values())

  min_ecc = min(ecc.values())
  logits = np.array([-(ecc[v] - min_ecc) / tau for v in infected_set])
  weights = np.exp(logits)
  weights /= weights.sum()

  for v, w in zip(infected_set, weights):
    post[v] = w
  return post

In [ ]:
def compute_rumor_posterior(G, infected_set, temp=2.5):
  """
  Shah-zaman Rumor Centrality - in how many different orders could the infection have
  spread if it started at v? the more possible ordering the more likely v is the source
  Rumor centrality assigns a score to every node in the infected subgraph.
  The node with the highest score is identified as the estimated source
  (the "rumor center").

  Since our network has cycles, we use build a BFS tree from each candidate root:
  keeping only the first edges that reaches each node.
  """
  G_I = G.subgraph(infected_set)
  post = np.zeros(len(G))
  if len(infected_set) == 1:
    only_node = next(iter(infected_set))
    post[only_node] = 1.0
    return post

  log_scores = {}
  for root in infected_set:
    bfs_edges = list(nx.bfs_edges(G_I, root))
    children = {u: [] for u in infected_set}
    order = [root]
    for u, v in bfs_edges:
      children[u].append(v)
      order.append(v)

    subtree_size = {u: 1 for u in infected_set}
    for u in reversed(order):
      for ch in children[u]:
        subtree_size[u] += subtree_size[ch]

    log_scores[root] = -sum(np.log(subtree_size[u]) / temp for u in infected_set)

  max_log = max(log_scores.values())
  weights = np.array([np.exp(log_scores[v] - max_log) for v in infected_set])
  weights /= weights.sum()

  for v, w in zip(infected_set, weights):
    post[v] = w
  return post


In [ ]:
def evaluate_seir(case='BA', sigmas=(1.0, 0.5, 0.2), gamma=0.1, modes=('I', 'IR', 'labels'),
                  n_runs=20, beta=0.3, frac=0.3, n_trials=20, tau=0.6):
  for sigma in sigmas:
    ranks, src_vis, hidden = {}, [], []
    for r in range(n_runs):
      G, _, _, hub = generate_topology(case=case, n=85, seed=r)
      n = len(G)
      adj = [list(G.neighbors(u)) for u in range(n)]
      D = dict(nx.all_pairs_shortest_path_length(G))
      rng = np.random.default_rng(7000 + r)

      st = None
      for _ in range(20):
        src = int(rng.integers(n))
        st = simulate_seir(adj, n, src, beta, sigma, gamma, frac, rng)
        if st is not None:
          break
      if st is None:
        continue
      obs_I = set(np.flatnonzero(st == I_).tolist())
      obs_R = set(np.flatnonzero(st == R_).tolist())
      if not obs_I:
        continue
      IR = obs_I | obs_R
      n_E = int((st == E_).sum())
      src_vis.append(src in obs_I)
      hidden.append(n_E / (n_E + len(IR)))

      for mode in modes:
        cand = list(range(n)) if mode == 'I' else sorted(IR)
        posts = {}
        if mode == 'I':
          ecc = np.array([max(D[v][u] for u in obs_I) for v in cand])
          w = np.exp(-(ecc - ecc.min()) / tau)
          p = np.zeros(n); p[cand] = w / w.sum()
          posts['jordan'] = p
        else:
          posts['jordan'] = jordan_posterior(G, IR)
          posts['rumor'] = compute_rumor_posterior(G, IR)
        posts['abc'] = abc_seir(adj, n, obs_I, obs_R, mode, cand, beta, sigma, gamma,
                                frac, n_trials, rng)
        rnd = np.zeros(n); rnd[cand] = 1.0 / len(cand)
        posts['random'] = rnd
        for k, post in posts.items():
          ranks.setdefault((mode, k), []).append(tie_rank(post, src))

    print(f"\n{case}  sigma={sigma} (latent ~{1/sigma:.0f} steps)  gamma={gamma}  runs={len(src_vis)}"
          f"  source still infectious: {np.mean(src_vis):.2f}  hidden in E: {np.mean(hidden):.2f}")
    for mode in modes:
      line = "  ".join(f"{k}: {np.median(ranks[(mode, k)]):5.1f}"
                       for k in ['random', 'jordan', 'rumor', 'abc'] if (mode, k) in ranks)
      print(f"  mode={mode:6s} median rank  {line}")


In [ ]:
evaluate_seir('BA')


BA  sigma=1.0 (latent ~1 steps)  gamma=0.1  runs=20  source still infectious: 0.45  hidden in E: 0.25
  mode=I      median rank  random:  43.0  jordan:  14.2  abc:   7.5
  mode=IR     median rank  random:  15.0  jordan:   5.0  rumor:  10.5  abc:   6.5
  mode=labels median rank  random:  15.0  jordan:   5.0  rumor:  10.5  abc:   4.5

BA  sigma=0.5 (latent ~2 steps)  gamma=0.1  runs=20  source still infectious: 0.30  hidden in E: 0.34
  mode=I      median rank  random:  43.0  jordan:  17.5  abc:   7.0
  mode=IR     median rank  random:  15.0  jordan:   8.2  rumor:  10.0  abc:   4.0
  mode=labels median rank  random:  15.0  jordan:   8.2  rumor:  10.0  abc:   4.5

BA  sigma=0.2 (latent ~5 steps)  gamma=0.1  runs=20  source still infectious: 0.10  hidden in E: 0.45
  mode=I      median rank  random:  43.0  jordan:  32.8  abc:  16.0
  mode=IR     median rank  random:  13.5  jordan:  12.2  rumor:   6.5  abc:   5.0
  mode=labels median rank  random:  13.5  jordan:  12.2  rumor:   6.5  abc:  

In [ ]:
evaluate_seir('WS')


WS  sigma=1.0 (latent ~1 steps)  gamma=0.1  runs=20  source still infectious: 0.25  hidden in E: 0.13
  mode=I      median rank  random:  43.0  jordan:   7.8  abc:   5.0
  mode=IR     median rank  random:  14.0  jordan:   2.2  rumor:   3.5  abc:   3.5
  mode=labels median rank  random:  14.0  jordan:   2.2  rumor:   3.5  abc:   4.0

WS  sigma=0.5 (latent ~2 steps)  gamma=0.1  runs=20  source still infectious: 0.20  hidden in E: 0.17
  mode=I      median rank  random:  43.0  jordan:   6.5  abc:   6.0
  mode=IR     median rank  random:  13.5  jordan:   4.0  rumor:   3.5  abc:   4.0
  mode=labels median rank  random:  13.5  jordan:   4.0  rumor:   3.5  abc:   4.0

WS  sigma=0.2 (latent ~5 steps)  gamma=0.1  runs=20  source still infectious: 0.05  hidden in E: 0.26
  mode=I      median rank  random:  43.0  jordan:   7.0  abc:   5.5
  mode=IR     median rank  random:  13.5  jordan:   3.2  rumor:   5.2  abc:   5.0
  mode=labels median rank  random:  13.5  jordan:   3.2  rumor:   5.2  abc:  

In [ ]:
'''
Latency is the waiting period between catching an infection and being able to spread.
So time spent in E

In our model each step an exposed node becomes infections with prob sigma,
So avg lantency - 1/sigma, the smaller the sigma the longer the latency


'''

In [ ]:
evaluate_seir('BA', gamma=0.3)


BA  sigma=1.0 (latent ~1 steps)  gamma=0.3  runs=20  source still infectious: 0.10  hidden in E: 0.21
  mode=I      median rank  random:  43.0  jordan:  20.5  abc:  31.5
  mode=IR     median rank  random:  14.5  jordan:  11.5  rumor:  11.8  abc:   6.5
  mode=labels median rank  random:  14.5  jordan:  11.5  rumor:  11.8  abc:   5.0

BA  sigma=0.5 (latent ~2 steps)  gamma=0.3  runs=20  source still infectious: 0.00  hidden in E: 0.31
  mode=I      median rank  random:  43.0  jordan:  43.8  abc:  31.5
  mode=IR     median rank  random:  14.5  jordan:  12.0  rumor:  10.5  abc:   8.5
  mode=labels median rank  random:  14.5  jordan:  12.0  rumor:  10.5  abc:   5.0

BA  sigma=0.2 (latent ~5 steps)  gamma=0.3  runs=20  source still infectious: 0.00  hidden in E: 0.38
  mode=I      median rank  random:  43.0  jordan:  37.5  abc:  31.5
  mode=IR     median rank  random:  13.5  jordan:  11.0  rumor:  10.2  abc:   7.5
  mode=labels median rank  random:  13.5  jordan:  11.0  rumor:  10.2  abc:  

In [ ]:
'''
With fast recovery the source is never visibly sick up to 38% of infections are hidden.
Current-case data then cant locate the source but knowing who has recovered still lets
simulation-based inference rank it about 4th-4th.
'''